# Debiasing: Oversampling Only (no GroupDRO)

Simple oversampling of small city groups + sqrt reweighting, WITHOUT GroupDRO.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

df_train = pd.read_csv(PROCESSED_DIR / "train.csv")
df_val = pd.read_csv(PROCESSED_DIR / "val.csv")
df_test = pd.read_csv(PROCESSED_DIR / "test.csv")

mapping = pd.read_csv(PROCESSED_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 16530, Val: 5510, Test: 5510


In [3]:
le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])

num_labels = len(le.classes_)
print(f"Num labels: {num_labels}")

Num labels: 9


In [4]:
# Oversampling small cities
MIN_CITY_SAMPLES = 100
OVERSAMPLE_FACTOR = 3  # More aggressive than before

city_counts = df_train["city_group"].value_counts()
small_cities = city_counts[city_counts < MIN_CITY_SAMPLES].index.tolist()

print(f"Small cities: {len(small_cities)}")

df_train_oversampled = df_train.copy()
for city in small_cities:
    city_samples = df_train[df_train["city_group"] == city]
    for _ in range(OVERSAMPLE_FACTOR - 1):
        df_train_oversampled = pd.concat([df_train_oversampled, city_samples], ignore_index=True)

print(f"Original: {len(df_train)}, After oversampling: {len(df_train_oversampled)}")

Small cities: 19
Original: 16530, After oversampling: 19304


In [5]:
# Sqrt reweighting
city_counts_new = df_train_oversampled["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts_new)
norm_w = raw_w / raw_w.mean()
city_weight_map = norm_w.to_dict()

df_train_oversampled["sample_weight"] = df_train_oversampled["city_group"].map(city_weight_map).astype(float)
print(f"Weight range: {df_train_oversampled['sample_weight'].min():.3f} - {df_train_oversampled['sample_weight'].max():.3f}")

Weight range: 0.199 - 1.443


In [6]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["resume_text"], padding="max_length", truncation=True, max_length=128)

In [7]:
train_dataset = Dataset.from_pandas(df_train_oversampled[["resume_text", "y", "sample_weight"]])
val_dataset = Dataset.from_pandas(df_val[["resume_text", "y"]])
test_dataset = Dataset.from_pandas(df_test[["resume_text", "y"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
test_dataset = test_dataset.rename_column("y", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Train size: {len(train_dataset)}")

Map: 100%|██████████| 5510/5510 [00:17<00:00, 309.78 examples/s]

Train size: 19304


In [8]:
# Simple weighted trainer (no GroupDRO)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs.get("labels")
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(logits, labels)
        
        if sample_weight is not None:
            per_sample_loss = per_sample_loss * sample_weight.to(per_sample_loss.dtype)
        
        loss = per_sample_loss.mean()
        return (loss, outputs) if return_outputs else loss

In [9]:
MODEL_NAME = "bert_oversample_only"

training_args = TrainingArguments(
    output_dir=f"./models/{MODEL_NAME}",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training: Oversampling + sqrt_reweight (NO GroupDRO)")
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1147.74it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Starting training: Oversampling + sqrt_reweight (NO GroupDRO)


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.621126,1.123098,0.596007,0.615370
2,0.594911,1.109555,0.607804,0.620866


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.10s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

TrainOutput(global_step=4826, training_loss=0.7381607593958476, metrics={'train_runtime': 11567.0585, 'train_samples_per_second': 3.338, 'train_steps_per_second': 0.417, 'total_flos': 2539707517366272.0, 'train_loss': 0.7381607593958476, 'epoch': 2.0})

In [11]:
predictions = trainer.predict(test_dataset)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(f"\n{'='*60}")
print("TEST RESULTS")
print(f"{'='*60}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



TEST RESULTS
Accuracy: 0.6005
Macro F1: 0.6115


In [12]:
# Fairness eval
df_eval = df_test.copy()
df_eval["y_true"] = y_true
df_eval["y_pred"] = y_pred
df_eval["correct"] = df_eval["y_true"] == df_eval["y_pred"]

def ovr_rates_by_group(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, g in enumerate(groups):
        df_g = df[df[group_col] == g]
        yt, yp = df_g["y_true"].values, df_g["y_pred"].values
        for c in range(num_classes):
            pos_mask = (yt == c)
            TP = np.sum((yp == c) & pos_mask)
            FN = np.sum((yp != c) & pos_mask)
            support[gi, c] = np.sum(pos_mask)
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return tpr, support, groups

def summarize_gaps(tpr, support=None, min_support=0):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[:, c]
        if support is not None and min_support > 0:
            col = col[support[:, c] >= min_support]
        col = col[~np.isnan(col)]
        gaps.append(np.max(col) - np.min(col) if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return np.max(valid) if len(valid) > 0 else np.nan, np.mean(valid) if len(valid) > 0 else np.nan

tpr, support, groups = ovr_rates_by_group(df_eval, "city_group", num_labels)
tpr_worst_robust, tpr_macro_robust = summarize_gaps(tpr, support, min_support=30)

print(f"\nFAIRNESS (robust, n>=30):")
print(f"  TPR Gap worst: {tpr_worst_robust:.4f}")
print(f"  TPR Gap macro: {tpr_macro_robust:.4f}")
print(f"\nCompare with best (0.609 acc, 0.329 worst, 0.116 macro)")


FAIRNESS (robust, n>=30):
  TPR Gap worst: 0.3583
  TPR Gap macro: 0.1349

Compare with best (0.609 acc, 0.329 worst, 0.116 macro)


In [13]:
# Save
SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")

config = {
    "method": "oversampling + sqrt_reweight (NO GroupDRO)",
    "oversample_factor": OVERSAMPLE_FACTOR,
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "tpr_gap_worst_robust": float(tpr_worst_robust),
    "tpr_gap_macro_robust": float(tpr_macro_robust)
}
with open(SAVE_DIR / "training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved to {SAVE_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]

Saved to models/bert_oversample_only
